In [1]:
import pandas as pd
csv_path = "/Users/hiraokatatsuru/Library/Mobile Documents/com~apple~CloudDocs/postal-operation-shift-management-system/db/init/csv/postal_datas.csv"
df_raw = pd.read_csv(csv_path)
df_raw.head()

,日付,通常郵便,書留,ゆうパケット,レターパックライト,レターパックプラス,特定記録,ゆうパック,eパケット,EMS,年賀組立,年賀配達
0,2021/10/1,63000,1783,882,615,264,588,1126,200,150,0,0
1,2021/10/2,0,1951,673,337,144,0,1534,0,158,0,0
2,2021/10/3,0,1054,898,151,65,0,1177,0,142,0,0
3,2021/10/4,102000,540,688,369,158,557,1595,400,274,0,0
4,2021/10/5,45000,721,877,443,190,451,1083,150,179,0,0


In [2]:
df = df_raw[["日付", "書留", "レターパックプラス"]].copy()
df = df.rename(columns={"日付": "date"})
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# registered_plus の目的変数
df["y"] = df["書留"].fillna(0) + df["レターパックプラス"].fillna(0)

df[["date", "y"]].head()

,date,y
0,2021-10-01,2047
1,2021-10-02,2095
2,2021-10-03,1119
3,2021-10-04,698
4,2021-10-05,911


In [3]:
import jpholiday

df_feat = df[["date", "y"]].copy()

df_feat["weekday"] = df_feat["date"].dt.weekday
df_feat["month"]   = df_feat["date"].dt.month
df_feat["year"]    = df_feat["date"].dt.year

df_feat["is_holiday"]  = df_feat["date"].dt.date.apply(jpholiday.is_holiday).astype(int)
df_feat["is_saturday"] = (df_feat["weekday"] == 5).astype(int)
df_feat["is_sunday"]   = (df_feat["weekday"] == 6).astype(int)

df_feat["is_day_before_holiday"] = (
    df_feat["date"].shift(-1).dt.date.apply(jpholiday.is_holiday).fillna(False).astype(int)
)

df_feat["is_nenmatsu"] = ((df_feat["month"] == 12) & (df_feat["date"].dt.day >= 25)).astype(int)
df_feat["is_obon"]     = ((df_feat["month"] == 8) & (df_feat["date"].dt.day.between(13, 16))).astype(int)


In [4]:
train_end = pd.Timestamp("2024-09-30")
valid_end = pd.Timestamp("2024-12-31")

train_mask = df_feat["date"] <= train_end

weekday_mean = (
    df_feat.loc[train_mask].groupby("weekday")["y"].mean().rename("weekday_mean_volume")
)
month_mean = (
    df_feat.loc[train_mask].groupby("month")["y"].mean().rename("month_mean_volume")
)

df_feat = df_feat.merge(weekday_mean, on="weekday", how="left")
df_feat = df_feat.merge(month_mean,   on="month",   how="left")


In [5]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

feature_cols = [
    "weekday","month","year",
    "is_holiday","is_saturday","is_sunday",
    "is_day_before_holiday","is_nenmatsu","is_obon",
    "weekday_mean_volume","month_mean_volume",
]

is_train = df_feat["date"] <= train_end
is_valid = (df_feat["date"] > train_end) & (df_feat["date"] <= valid_end)
is_test  = df_feat["date"] > valid_end

X = df_feat[feature_cols]
y = df_feat["y"]

X_train, y_train = X.loc[is_train], y.loc[is_train]
X_valid, y_valid = X.loc[is_valid], y.loc[is_valid]
X_test,  y_test  = X.loc[is_test],  y.loc[is_test]

model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

pred_train = model.predict(X_train)
pred_valid = model.predict(X_valid)

print("MAE train:", mean_absolute_error(y_train, pred_train))
print("MAE valid:", mean_absolute_error(y_valid, pred_valid))

if len(X_test) > 0:
    pred_test = model.predict(X_test)
    print("MAE test :", mean_absolute_error(y_test, pred_test))


MAE train: 132.15391825933526
MAE valid: 149.55333212147588
MAE test : 587.9572143554688


In [6]:
pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

weekday_mean_volume      0.485371
is_obon                  0.116477
weekday                  0.103521
is_nenmatsu              0.093718
is_holiday               0.041745
month_mean_volume        0.036891
is_saturday              0.031671
year                     0.023476
is_sunday                0.022987
month                    0.022815
is_day_before_holiday    0.021327
dtype: float32

In [7]:
forecast_start = pd.Timestamp("2025-01-16")
days = 28

df_future = pd.DataFrame({"date": pd.date_range(forecast_start, periods=days, freq="D")})

# 未来特徴量（同じロジック）
df_future_feat = df_future.copy()
df_future_feat["weekday"] = df_future_feat["date"].dt.weekday
df_future_feat["month"]   = df_future_feat["date"].dt.month
df_future_feat["year"]    = df_future_feat["date"].dt.year

df_future_feat["is_holiday"]  = df_future_feat["date"].dt.date.apply(jpholiday.is_holiday).astype(int)
df_future_feat["is_saturday"] = (df_future_feat["weekday"] == 5).astype(int)
df_future_feat["is_sunday"]   = (df_future_feat["weekday"] == 6).astype(int)

df_future_feat["is_day_before_holiday"] = (
    df_future_feat["date"].shift(-1).dt.date.apply(jpholiday.is_holiday).fillna(False).astype(int)
)
df_future_feat["is_nenmatsu"] = ((df_future_feat["month"] == 12) & (df_future_feat["date"].dt.day >= 25)).astype(int)
df_future_feat["is_obon"]     = ((df_future_feat["month"] == 8) & (df_future_feat["date"].dt.day.between(13, 16))).astype(int)

# train由来の平均を適用（再計算しない）
df_future_feat = df_future_feat.merge(weekday_mean, on="weekday", how="left")
df_future_feat = df_future_feat.merge(month_mean,   on="month",   how="left")

df_future_feat["forecast_raw"] = model.predict(df_future_feat[feature_cols])

df_forecast = df_future_feat[["date", "forecast_raw"]].rename(columns={"forecast_raw": "forecast_volume"})
df_forecast.head()


,date,forecast_volume
0,2025-01-16,1500.917969
1,2025-01-17,1420.360962
2,2025-01-18,1458.060303
3,2025-01-19,1291.280518
4,2025-01-20,656.802612


In [11]:
df_future_feat["forecast_volume"] = (
    df_future_feat["forecast_raw"]
    .astype("float64")
    .round()
    .astype(int)
)



In [10]:
df_future_feat["forecast_raw"] = model.predict(
    df_future_feat[feature_cols]
)



In [12]:
df_result = df_future_feat[[
    "date",
    "forecast_volume"
]].copy()

df_result


,date,forecast_volume
0,2025-01-16,1501
1,2025-01-17,1420
2,2025-01-18,1458
3,2025-01-19,1291
4,2025-01-20,657
5,2025-01-21,974
6,2025-01-22,1299
7,2025-01-23,1501
8,2025-01-24,1420
9,2025-01-25,1458


In [13]:
from posms.models_lagless.registered_plus import train_and_forecast_28d_from_csv

out, metrics = train_and_forecast_28d_from_csv(
    csv_path,
    forecast_start="2025-01-16",
    days=28,
    train_end="2024-09-30",
    valid_end="2024-12-31",
)

metrics
out  # 28行すべて表示


,date,forecast_volume
0,2025-01-16,1501
1,2025-01-17,1420
2,2025-01-18,1458
3,2025-01-19,1291
4,2025-01-20,657
5,2025-01-21,974
6,2025-01-22,1299
7,2025-01-23,1501
8,2025-01-24,1420
9,2025-01-25,1458
